# KCGS 평가등급 ↔ 자동채점 점수 상관 EDA**목적** — RAG 기반 K-ESG 자동채점 결과가 외부 ESG 평가등급(KCGS)과 얼마나 일치하는지 검증한다.**데이터** — `[공동작업] KCGS_평가등급_2022-2025_채점결과.xlsx` 의 `입력 모델 점수 결과` 시트· 54개 기업 × 2022~2025 = 171개 기업-연도 (등급·점수 모두 존재하는 행)**분석 순서**| # | 내용 | 답하려는 질문 ||---|---|---|| 1 | 데이터 품질 | 결측·중복·이상값이 있는가 || 2 | 기술통계 | 점수와 등급이 어떻게 분포하는가 || 3 | 상관 | 등급이 높을수록 점수가 높은가 || 4 | 등급별 평균·판별력 | 인접 등급을 갈라내는가 || 5 | 판별타당도 | E/S/G를 **구분해서** 재고 있는가 || 6 | 공시량 편향 | 점수가 보고서 발행 여부에 끌려가는가 || 7 | 기업내 변화 | 같은 기업의 연도별 변화를 따라가는가 || 8 | 회귀·불일치 사례 | 무엇이 점수를 설명하는가 / 어디가 틀렸는가 |

## 0. 환경 설정

In [ ]:
# 한글 폰트 (코랩 최초 1회, 약 30초). 실행 후 런타임 재시작 불필요.
!apt-get -qq install -y fonts-nanum > /dev/null
!fc-cache -f > /dev/null

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

fm._load_fontmanager(try_read_cache=False)
matplotlib.rc("font", family="NanumGothic")
matplotlib.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 110
print("폰트 설정 완료:", matplotlib.rcParams["font.family"])

In [ ]:
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)
print(pd.__version__, np.__version__)

## 1. 데이터 로드아래 두 방법 중 하나를 쓴다.- **A. 직접 업로드** — 셀 실행 후 파일 선택 (기본값)- **B. 구글 드라이브** — `USE_DRIVE = True` 로 바꾸고 `DRIVE_PATH` 수정

In [ ]:
USE_DRIVE = False
DRIVE_PATH = "/content/drive/MyDrive/[공동작업] KCGS_평가등급_2022-2025_채점결과.xlsx"
SHEET = "입력 모델 점수 결과"

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    XLSX = DRIVE_PATH
else:
    from google.colab import files
    up = files.upload()
    XLSX = list(up.keys())[0]

print("사용 파일:", XLSX)

In [ ]:
raw = pd.read_excel(XLSX, sheet_name=SHEET, dtype={"기업코드": str})
print("원본:", raw.shape)
raw.head(3)

## 2. 전처리 및 데이터 품질 점검

In [ ]:
ORD  = {"D": 1, "C": 2, "B": 3, "B+": 4, "A": 5, "A+": 6, "S": 7}   # KCGS 7단계
LAB  = {v: k for k, v in ORD.items()}
GCOLS = ["ESG등급", "환경", "사회", "지배구조"]
SCOLS = ["전체결과", "P결과", "E결과", "S결과", "G결과"]
# (표시명, 등급 컬럼, 점수 컬럼)
PAIRS = [("전체", "ESG등급", "전체결과"), ("환경(E)", "환경", "E결과"),
         ("사회(S)", "사회", "S결과"), ("지배구조(G)", "지배구조", "G결과")]

df = raw.copy()
df["기업코드"] = df["기업코드"].astype(str).str.replace(r"\.0$", "", regex=True).str.zfill(6)
df["기업명"]  = df["기업명"].replace({"디티알오토모티브": "DN오토모티브"})   # 사명 변경, 같은 코드
df["보고서_발행_유형"] = df["보고서_발행_유형"].str.strip().replace({"지속가능경영": "지속가능경영만"})

# 점수 컬럼: '상폐' 등 문자열이 섞여 있어 숫자만 남긴다
nonnum = df[df[SCOLS[0]].notna() & pd.to_numeric(df[SCOLS[0]], errors="coerce").isna()]
print("숫자가 아닌 점수 행:", len(nonnum))
display(nonnum[["기업명", "평가년도", "전체결과", "비고"]])

for c in SCOLS:
    df[c] = pd.to_numeric(df[c], errors="coerce")
for g in GCOLS:
    df[g + "_ord"] = df[g].map(ORD)

df = df[df["전체결과"].notna()].copy()
df["SR"] = df["지속가능경영보고서_발행"].fillna(0).astype(int)      # 지속가능경영보고서
df["CG"] = df["기업지배구조보고서_발행"].fillna(0).astype(int)      # 기업지배구조보고서
df["보고서수"] = df["SR"] + df["CG"]

print(f"\n분석 대상: {len(df)}개 기업-연도 / {df['기업코드'].nunique()}개 기업")

In [ ]:
print("■ 등급 매핑 실패")
for g in GCOLS:
    bad = df[df[g + "_ord"].isna()]
    print(f"  {g}: {len(bad)}건")

print("\n■ 점수 결측 / 범위(0~100) 이탈")
for c in SCOLS:
    print(f"  {c}: 결측 {df[c].isna().sum()}건, 범위밖 {((df[c] < 0) | (df[c] > 100)).sum()}건")

print("\n■ 중복 (기업코드 × 평가년도):", df.duplicated(["기업코드", "평가년도"]).sum(), "건")

print("\n■ 연도별 커버리지")
display(df.groupby("평가년도").agg(기업수=("기업코드", "nunique"), 행수=("전체결과", "size")))

print("■ 기업별 보유 연도 수")
display(df.groupby("기업코드").size().value_counts().sort_index().rename("기업 수").to_frame())

## 3. 기술통계 — 점수와 등급의 분포

In [ ]:
desc = df[SCOLS].describe().T
desc["IQR"] = desc["75%"] - desc["25%"]
desc["CV"]  = desc["std"] / desc["mean"]
display(desc[["count", "mean", "std", "min", "25%", "50%", "75%", "max", "IQR", "CV"]].round(2))

gd = pd.DataFrame({g: df[g].value_counts() for g in GCOLS}).reindex(list(ORD)[::-1]).fillna(0).astype(int)
display(gd)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].boxplot([df[c].dropna() for c in SCOLS], showmeans=True)
axes[0].set_xticklabels(SCOLS)
axes[0].set_title("채점 점수 분포")
axes[0].set_ylabel("점수 (0~100)")
axes[0].grid(axis="y", alpha=.3)

gd.plot(kind="bar", ax=axes[1], width=.8)
axes[1].set_title("KCGS 등급 분포")
axes[1].set_xlabel("등급")
axes[1].set_ylabel("건수")
axes[1].grid(axis="y", alpha=.3)

plt.tight_layout(); plt.show()

> **읽는 법** — E결과의 평균이 30.6으로 가장 낮고 변동계수(CV)가 0.57로 가장 크다.> G결과는 평균 50.8에 CV 0.25로 가장 안정적이다. 등급은 A·A+에 몰려 있어 상위 편중이다.

## 4. 등급 ↔ 점수 상관

In [ ]:
# 등급이 다른 모든 쌍 중 점수 순서가 맞는 비율 (Somers' D 계열)
def order_agreement(gvals, svals):
    a = np.asarray(gvals); b = np.asarray(svals)
    conc = disc = tie = 0
    for i, j in itertools.combinations(range(len(a)), 2):
        if a[i] == a[j]:
            continue
        gs, ss = np.sign(a[i] - a[j]), np.sign(b[i] - b[j])
        if ss == 0:   tie  += 1
        elif gs == ss: conc += 1
        else:          disc += 1
    tot = conc + disc + tie
    return conc / tot, disc / tot, (conc + .5 * tie) / tot, tot


rows = []
for name, g, s in PAIRS:
    d = df[[g + "_ord", s]].dropna()
    x, y = d[g + "_ord"], d[s]
    pr, sp, kt = stats.pearsonr(x, y), stats.spearmanr(x, y), stats.kendalltau(x, y)
    c, dsc, agr, tot = order_agreement(x, y)
    rows.append({"영역": name, "n": len(d),
                 "Pearson r": pr.statistic, "Spearman ρ": sp.statistic, "Kendall τ": kt.statistic,
                 "p(Spearman)": sp.pvalue, "순서일치": c, "순서역전": dsc, "순서일치도": agr})
corr_tbl = pd.DataFrame(rows).set_index("영역")
display(corr_tbl.round(4))

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(17, 4), sharey=False)
rng = np.random.default_rng(0)

for ax, (name, g, s) in zip(axes, PAIRS):
    d = df[[g + "_ord", s]].dropna()
    x = d[g + "_ord"] + rng.uniform(-.16, .16, len(d))      # 겹침 방지용 지터
    ax.scatter(x, d[s], s=18, alpha=.55, edgecolor="none")
    b, a = np.polyfit(d[g + "_ord"], d[s], 1)
    xs = np.linspace(d[g + "_ord"].min(), d[g + "_ord"].max(), 10)
    ax.plot(xs, b * xs + a, color="crimson", lw=1.6)
    m = d.groupby(g + "_ord")[s].mean()
    ax.plot(m.index, m.values, "o-", color="black", lw=1.2, ms=5, label="등급별 평균")
    rho = stats.spearmanr(d[g + "_ord"], d[s]).statistic
    ax.set_title(f"{name}  ρ={rho:.3f}")
    ax.set_xticks(sorted(d[g + "_ord"].unique()))
    ax.set_xticklabels([LAB[i] for i in sorted(d[g + "_ord"].unique())])
    ax.set_xlabel("KCGS 등급"); ax.grid(alpha=.3)
axes[0].set_ylabel("채점 점수"); axes[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# 연도별 안정성 + 하위표본별 상관
yr = pd.DataFrame([
    {"평가년도": y, "n": len(sub),
     **{name: stats.spearmanr(sub[g + "_ord"], sub[s]).statistic for name, g, s in PAIRS}}
    for y, sub in df.groupby("평가년도")]).set_index("평가년도")
print("■ 연도별 Spearman ρ"); display(yr.round(3))

subsets = [("전체 표본", df),
           ("보고서 둘 다 발행", df[(df.SR == 1) & (df.CG == 1)]),
           ("지속가능경영보고서 발행", df[df.SR == 1]),
           ("지속가능경영보고서 미발행", df[df.SR == 0]),
           ("KOSPI만", df[df.시장구분 == "KOSPI"])]
sub_tbl = pd.DataFrame([
    {"하위표본": lb, "n": len(sb),
     **{name: (stats.spearmanr(sb[g + "_ord"], sb[s]).statistic if len(sb) >= 8 else np.nan)
        for name, g, s in PAIRS}}
    for lb, sb in subsets]).set_index("하위표본")
print("■ 하위표본별 Spearman ρ"); display(sub_tbl.round(3))

# 기업 단위(연도 평균)
agg = df.groupby(["기업코드", "기업명"]).agg(
    {**{s: "mean" for _, _, s in PAIRS}, **{g + "_ord": "mean" for _, g, _ in PAIRS}}).reset_index()
print("■ 기업 단위 (연도 평균, n=%d)" % len(agg))
display(pd.DataFrame([
    {"영역": name, "Spearman ρ": stats.spearmanr(agg[g + "_ord"], agg[s]).statistic}
    for name, g, s in PAIRS]).set_index("영역").round(3))

> **결과** — 네 영역 모두 ρ≈0.62~0.68, p<1e-12. 등급이 다른 두 기업을 뽑았을 때 점수 순서가 맞는 비율 78~80%.> 기업 단위로 집계하면 ρ≈0.71~0.72로 더 오른다. **기업 간 서열은 잘 맞춘다.**> 다만 지속가능경영보고서 미발행 하위표본에서는 E(0.342)·G(0.224)가 유의성을 잃는다.

## 5. 등급별 평균과 인접 등급 판별력

In [ ]:
for name, g, s in PAIRS:
    t = df.groupby(g + "_ord")[s].agg(["count", "mean", "std", "min", "max"]).round(1)
    t.index = [LAB[i] for i in t.index]
    means = df.groupby(g + "_ord")[s].mean()
    inv = sum(means.iloc[k + 1] < means.iloc[k] for k in range(len(means) - 1))
    print(f"[{name}] 등급이 오를 때 평균이 내려간 구간: {inv}/{len(means)-1}")
    display(t)

In [ ]:
rows = []
for name, g, s in PAIRS:
    lv = sorted(df[g + "_ord"].dropna().unique())
    for a, b in zip(lv[:-1], lv[1:]):
        x, y = df.loc[df[g + "_ord"] == a, s], df.loc[df[g + "_ord"] == b, s]
        if len(x) < 3 or len(y) < 3:
            continue
        pooled = np.sqrt(((len(x)-1)*x.var() + (len(y)-1)*y.var()) / (len(x)+len(y)-2))
        auc = stats.mannwhitneyu(y, x).statistic / (len(x) * len(y))
        rows.append({"영역": name, "구간": f"{LAB[a]}→{LAB[b]}", "n(하)": len(x), "n(상)": len(y),
                     "Cohen d": (y.mean()-x.mean())/pooled, "AUC": auc,
                     "판정": "구분 못함" if auc < .6 else ("약함" if auc < .7 else "구분됨")})
adj = pd.DataFrame(rows)
display(adj.round(2).style.map(
    lambda v: "background-color:#fce4d6" if v == "구분 못함" else
              ("background-color:#e2efda" if v == "구분됨" else ""), subset=["판정"]))

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(17, 4))
for ax, (name, g, s) in zip(axes, PAIRS):
    lv = sorted(df[g + "_ord"].dropna().unique())
    ax.boxplot([df.loc[df[g + "_ord"] == l, s].dropna() for l in lv], showmeans=True)
    ax.set_xticklabels([LAB[l] for l in lv])
    ax.set_title(name); ax.set_xlabel("KCGS 등급"); ax.grid(axis="y", alpha=.3)
axes[0].set_ylabel("채점 점수")
plt.tight_layout(); plt.show()

> **결과** — 상자가 서로 크게 겹친다. 전체의 `B→B+` 판별 AUC 0.53(무작위 수준), 사회의 `C→B` 0.50.> 제대로 갈리는 구간은 `A→A+`(0.83)와 `C→B`(0.74) 정도로, **상·하위 극단만 잡고 중간은 뭉갠다.**

## 6. 판별타당도 — E/S/G를 구분해서 재고 있는가> 영역 점수는 **자기 영역 등급**과 가장 강하게 붙어야 한다.> 대각선이 서지 않으면 세 영역이 사실상 같은 것을 재고 있다는 뜻이다.

In [ ]:
cross = pd.DataFrame(
    {g: [stats.spearmanr(df[s], df[g + "_ord"]).statistic for s in SCOLS] for g in GCOLS},
    index=SCOLS)
display(cross.round(3).style.background_gradient(cmap="Blues", axis=None).format("{:.3f}"))

rows = []
for s, g in [("E결과", "환경"), ("S결과", "사회"), ("G결과", "지배구조")]:
    own = cross.loc[s, g]
    others = cross.loc[s, [c for c in GCOLS if c not in (g, "ESG등급")]]
    rows.append({"점수": s, "자기 등급": g, "ρ(자기)": own,
                 "최대 타영역": others.idxmax(), "ρ(타영역)": others.max(),
                 "차이": own - others.max()})
print("■ 자기 영역 우위 (차이가 0 근처면 구분 실패)")
display(pd.DataFrame(rows).set_index("점수").round(3))

print("■ 점수끼리의 상관 (높을수록 단일 축)")
display(df[SCOLS].corr(method="spearman").round(3)
        .style.background_gradient(cmap="Oranges", axis=None).format("{:.3f}"))

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4))
im = ax.imshow(cross.values, cmap="Blues", vmin=.4, vmax=.75)
ax.set_xticks(range(len(GCOLS)), GCOLS)
ax.set_yticks(range(len(SCOLS)), SCOLS)
for i in range(len(SCOLS)):
    for j in range(len(GCOLS)):
        ax.text(j, i, f"{cross.values[i, j]:.3f}", ha="center", va="center",
                color="white" if cross.values[i, j] > .63 else "black", fontsize=9)
ax.set_title("점수 × 등급 교차 상관 (Spearman ρ)")
plt.colorbar(im); plt.tight_layout(); plt.show()

> **결과 — 판별타당도 실패.**> E결과는 환경등급(0.623)과 사회등급(0.613)에 거의 똑같이 반응하고, S결과는 환경·사회에 **동률**(0.676/0.676)이다.> 점수끼리도 E↔S 0.877, 전체↔E 0.959로 붙어 있다. **사실상 하나의 축만 재고 있다.**> 자기 등급에 더 붙는 것은 G뿐(+0.062)이다.

## 7. 공시량 편향 — 점수가 보고서 발행 여부에 끌려가는가

In [ ]:
t = df.groupby(["SR", "CG"]).agg(n=("전체결과", "size"), KCGS등급평균=("ESG등급_ord", "mean"),
                                 전체=("전체결과", "mean"), P=("P결과", "mean"), E=("E결과", "mean"),
                                 S=("S결과", "mean"), G=("G결과", "mean")).round(2)
t.index = pd.MultiIndex.from_tuples(
    [(f"지속가능 {'O' if a else 'X'}", f"지배구조 {'O' if b else 'X'}") for a, b in t.index])
display(t)

print("■ 같은 KCGS 등급 안에서 지속가능경영보고서 유무가 만드는 점수 차")
rows = []
for name, g, s in PAIRS:
    parts, w, acc = [], 0, 0
    for lvl, sub in df.groupby(g + "_ord"):
        a, b = sub.loc[sub.SR == 1, s], sub.loc[sub.SR == 0, s]
        if len(a) >= 3 and len(b) >= 3:
            parts.append(f"{LAB[lvl]} {a.mean()-b.mean():+.1f}")
            acc += (len(a) + len(b)) * (a.mean() - b.mean()); w += len(a) + len(b)
    rows.append({"영역": name, "등급통제 평균차": acc / w if w else np.nan, "등급별 내역": ", ".join(parts)})
display(pd.DataFrame(rows).set_index("영역").round(1))

In [ ]:
print("■ 보고서 수와의 상관 비교 + 보고서 수를 통제한 편상관")
rows = []
for name, g, s in PAIRS:
    d  = df[[g + "_ord", s, "보고서수"]].dropna()
    rk = d.rank()
    res_s = rk[s]          - np.polyval(np.polyfit(rk["보고서수"], rk[s], 1), rk["보고서수"])
    res_g = rk[g + "_ord"] - np.polyval(np.polyfit(rk["보고서수"], rk[g + "_ord"], 1), rk["보고서수"])
    rows.append({"영역": name,
                 "등급↔점수 ρ": stats.spearmanr(d[g + "_ord"], d[s]).statistic,
                 "보고서수↔점수 ρ": stats.spearmanr(d["보고서수"], d[s]).statistic,
                 "보고서수↔등급 ρ": stats.spearmanr(d["보고서수"], d[g + "_ord"]).statistic,
                 "편상관(보고서수 통제)": stats.pearsonr(res_g, res_s)[0]})
part = pd.DataFrame(rows).set_index("영역")
part["감소폭"] = part["등급↔점수 ρ"] - part["편상관(보고서수 통제)"]
display(part.round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

grp = df.groupby("보고서_발행_유형")[["E결과", "S결과", "G결과"]].mean()
grp.plot(kind="bar", ax=axes[0], width=.8)
axes[0].set_title("보고서 발행 유형별 영역 점수")
axes[0].set_ylabel("평균 점수"); axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=0); axes[0].grid(axis="y", alpha=.3)

for sr, lbl, mk in [(1, "지속가능경영보고서 발행", "o"), (0, "미발행", "^")]:
    d = df[df.SR == sr]
    axes[1].scatter(d["ESG등급_ord"] + np.random.uniform(-.15, .15, len(d)),
                    d["전체결과"], s=22, alpha=.6, marker=mk, label=lbl)
axes[1].set_xticks(sorted(df["ESG등급_ord"].dropna().unique()),
                   [LAB[i] for i in sorted(df["ESG등급_ord"].dropna().unique())])
axes[1].set_xlabel("KCGS ESG등급"); axes[1].set_ylabel("전체 채점점수")
axes[1].set_title("같은 등급이라도 보고서 유무로 갈린다"); axes[1].legend(); axes[1].grid(alpha=.3)

plt.tight_layout(); plt.show()

> **결과** — 같은 KCGS 등급 안에서도 지속가능경영보고서를 내면 **전체 +17.8점 / E +18.2점 / S +17.3점**을 더 받는다.> 미발행 28개사의 E점수 평균은 8.0점(최대 34.1)으로 사실상 채점 불가 상태다.> 보고서 수를 통제하면 전체 ρ가 0.679 → 0.517로 떨어진다.> **G만 예외**로 등급통제 차이가 +5.7점에 그치는데, 지배구조보고서가 의무공시라 조건이 균등하기 때문이다.

## 8. 회귀 — 무엇이 점수를 설명하는가

In [ ]:
def ols(y, X):
    X1 = np.column_stack([np.ones(len(X)), X])
    beta, *_ = np.linalg.lstsq(X1, y, rcond=None)
    r2 = 1 - ((y - X1 @ beta) ** 2).sum() / ((y - y.mean()) ** 2).sum()
    adj = 1 - (1 - r2) * (len(y) - 1) / (len(y) - X1.shape[1])
    return beta, r2, adj


d = df.dropna(subset=["전체결과", "ESG등급_ord"])
y = d["전체결과"].to_numpy()
rows = []
for label, cols in [("등급만", ["ESG등급_ord"]),
                    ("공시량만 (SR, CG)", ["SR", "CG"]),
                    ("등급 + 공시량", ["ESG등급_ord", "SR", "CG"])]:
    beta, r2, adj = ols(y, d[cols].to_numpy(float))
    rows.append({"모형": label, "R²": r2, "adj R²": adj, "절편": beta[0],
                 "계수": ", ".join(f"{c}={b:+.2f}" for c, b in zip(cols, beta[1:]))})
display(pd.DataFrame(rows).set_index("모형").round(3))

> **결과** — 등급만 넣은 모형(R²=0.425)보다 **보고서 발행 여부만 넣은 모형(R²=0.510)이 점수를 더 잘 설명한다.**> 지금 점수가 재고 있는 것은 ESG 성과가 아니라 '공시 충실도'에 가깝다.

## 9. 기업 내 변화 추적 — 같은 기업의 연도별 개선을 잡아내는가

In [ ]:
multi = df[df.groupby("기업코드")["기업코드"].transform("size") >= 2]
print(f"2개 연도 이상 보유: {multi['기업코드'].nunique()}개 기업 / {len(multi)}행\n")

rows = []
for name, g, s in PAIRS:
    d = multi[[g + "_ord", s, "기업코드"]].dropna().copy()
    d["g_dm"] = d[g + "_ord"] - d.groupby("기업코드")[g + "_ord"].transform("mean")
    d["s_dm"] = d[s]          - d.groupby("기업코드")[s].transform("mean")
    wr = stats.pearsonr(d["g_dm"], d["s_dm"])                       # 기업 고정효과 제거 후

    e = df.sort_values(["기업코드", "평가년도"]).copy()               # 1차 차분
    e["dg"] = e.groupby("기업코드")[g + "_ord"].diff()
    e["ds"] = e.groupby("기업코드")[s].diff()
    ee = e.dropna(subset=["dg", "ds"])
    mv = ee[ee.dg != 0]
    same = np.sign(mv.dg) == np.sign(mv.ds)
    dr = stats.pearsonr(ee["dg"], ee["ds"])

    rows.append({"영역": name, "기업내 r": wr.statistic, "p": wr.pvalue,
                 "Δ상관 r": dr.statistic, "p ": dr.pvalue,
                 "등급 변동 건수": len(mv), "방향 일치": int(same.sum()), "일치율": same.mean()})
display(pd.DataFrame(rows).set_index("영역").round(3))

> **결과** — 기업내 상관 r=0.18~0.38, Δ등급↔Δ점수 상관은 r=0.05~0.17로 **모두 유의하지 않다**(p>0.05).> 등급이 실제로 움직인 57건 중 점수가 같은 방향으로 움직인 비율은 58~63%로 동전던지기와 큰 차이가 없다.> **개선 추적 도구로는 아직 쓸 수 없다.**

## 10. 불일치 사례

In [ ]:
d = df.dropna(subset=["ESG등급_ord", "전체결과"]).copy()
b, a = np.polyfit(d["ESG등급_ord"], d["전체결과"], 1)
d["기대점수"] = b * d["ESG등급_ord"] + a
d["잔차"]   = d["전체결과"] - d["기대점수"]
d["z"]     = d["잔차"] / d["잔차"].std()
cols = ["기업명", "평가년도", "ESG등급", "전체결과", "기대점수", "잔차", "z", "보고서_발행_유형"]

print("■ 모델이 과대평가 (등급 대비 점수 높음) 상위 10")
display(d.nlargest(10, "z")[cols].round(1))
print("■ 모델이 과소평가 하위 10")
display(d.nsmallest(10, "z")[cols].round(1))

In [ ]:
agg2 = agg.copy()
agg2["점수순위"] = agg2["전체결과"].rank(ascending=False)
agg2["등급순위"] = agg2["ESG등급_ord"].rank(ascending=False)
agg2["순위차"]  = agg2["점수순위"] - agg2["등급순위"]

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(agg2["등급순위"], agg2["점수순위"], s=30, alpha=.7)
ax.plot([0, 55], [0, 55], "--", color="gray", lw=1)
for _, r in pd.concat([agg2.nlargest(5, "순위차"), agg2.nsmallest(5, "순위차")]).iterrows():
    ax.annotate(r["기업명"], (r["등급순위"], r["점수순위"]), fontsize=8,
                xytext=(4, 3), textcoords="offset points")
ax.set_xlabel("KCGS 등급 순위 (1위 = 최상위)")
ax.set_ylabel("채점 점수 순위")
ax.set_title("기업 단위 순위 비교 (n=54)\n대각선 위 = 모델이 과소평가, 아래 = 과대평가")
ax.grid(alpha=.3); plt.tight_layout(); plt.show()

display(agg2.sort_values("순위차")[["기업명", "ESG등급_ord", "전체결과",
                                   "등급순위", "점수순위", "순위차"]].round(2).head(30))

## 11. 결론**1. 상관은 통계적으로 분명하다 — 다만 '기업 간 서열'에 한정된다**- 전체 ρ=0.679, E 0.623, S 0.676, G 0.658 (모두 p<1e-12)- 등급이 다른 두 기업의 점수 순서가 맞는 비율 78~80%- 기업 단위(연도 평균, n=54)로는 네 영역 모두 ρ≈0.71~0.72**2. 그러나 E/S/G를 구분하지 못한다 (판별타당도 실패)**- E점수 ↔ 환경등급 0.623 ≈ 사회등급 0.613- S점수 ↔ 사회등급 0.676 **=** 환경등급 0.676- 점수끼리 E↔S 0.877, 전체↔E 0.959 → 사실상 단일 축- 자기 등급에 더 붙는 건 **G뿐**(+0.062)**3. 점수를 가장 크게 좌우하는 건 등급이 아니라 보고서 발행 여부**- 회귀 R²: 등급만 0.425 < **공시량만 0.510**- 등급 통제 후에도 지속가능경영보고서 발행 시 전체 +17.8점, E +18.2점- 미발행 28개사의 E점수 평균 8.0점 (최대 34.1)- 보고서 수 통제 시 ρ 0.679 → 0.517- **G는 예외**(+5.7점) — 지배구조보고서가 의무공시라 조건이 균등**4. 같은 기업의 연도별 변화는 못 따라간다**- 기업내 r=0.18~0.38, Δ상관 r=0.05~0.17 (모두 유의하지 않음)- 등급 변동 57건 중 방향 일치 58~63%**5. 중간 등급대를 못 가른다**- 전체 `B→B+` AUC 0.53, 사회 `C→B` AUC 0.50 (무작위 수준)- 잘 갈리는 구간은 `A→A+`(0.83), `C→B`(0.74)뿐---### 종합지금 모델은 **"보고서를 성실히 낸 대기업"을 골라내는 스크리너**로는 작동하지만,E/S/G를 구분해 측정하거나 같은 기업의 개선을 추적하는 도구로는 아직 작동하지 않는다.### 다음 단계| # | 할 일 | 근거 ||---|---|---|| ① | 지속가능경영보고서 미발행 기업을 '측정 불가'로 분리하거나 별도 보정 | 등급 통제 후에도 +17.8점 차이 || ② | E/S의 정량 표 추출 실패부터 수정 | 1차 검수: E 채점기준 되풀이 146건, S 환각 168건 || ③ | G는 기준선(control)으로 유지 | 공시 조건 균등 + 유일하게 판별타당도 확보 || ④ | 연도 변화 추적이 목표라면 현 파이프라인으로는 불가 | Δ상관 전 영역 유의하지 않음 |

## 부록 — 분석 데이터 내려받기

In [ ]:
keep = ["기업명", "기업코드", "평가년도", "ESG등급", "환경", "사회", "지배구조",
        "ESG등급_ord", "환경_ord", "사회_ord", "지배구조_ord",
        "전체결과", "P결과", "E결과", "S결과", "G결과",
        "시장구분", "보고서_발행_유형", "SR", "CG", "보고서수", "시가총액"]
out = df[keep].sort_values(["기업코드", "평가년도"])
out.to_csv("EDA_분석데이터.csv", index=False, encoding="utf-8-sig")
print(out.shape)

# 코랩에서 내려받기
# from google.colab import files; files.download("EDA_분석데이터.csv")
out.head()